# Follow-up 实验：OpenAI 官方 API

这份 Notebook **仅连接 OpenAI 官方接口**，不再经过中转平台。

使用方式：

1. 运行 key 单元，在隐藏输入框中粘贴原生 OpenAI API key。
2. 运行定义单元。
3. 在最后一个单元只填写 MODEL_NAME 和 SAMPLE_COUNT，然后运行。

说明：

- 官方端点固定为 https://api.openai.com/v1/chat/completions。
- key 只保存在当前内核内存并通过子进程环境变量传递，不写入 Notebook、命令行、结果或元数据。
- 程序依次执行离线计划、3 次冒烟测试和正式实验；冒烟失败时不会开始正式实验。
- 相同模型、样本数和实验配置重新运行时，会跳过已有成功记录并断点续跑。
- 结果写入 runs/openai/<MODEL_NAME>@official/，与中转平台结果完全分开。
- 100/300 人使用仓库中的固定患者名单；其他人数按项目固定随机种子抽样。


In [ ]:
import os
from getpass import getpass

# 密钥从环境变量或隐藏输入读取，不写入 Notebook。
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import uuid
from contextlib import contextmanager
from pathlib import Path

import yaml


OPENAI_CHAT_COMPLETIONS_URL = "https://api.openai.com/v1/chat/completions"
CONDITIONS = "main"
MAIN_CONDITION_COUNT = 12
REPEATS = 5
MONTH = 12
WORKERS = 8
SMOKE_CALLS = 3
MAX_COMPLETION_TOKENS = 2000
SENTINEL_EVERY = 1500
SENTINEL_CONDITION_COUNT = 3
SENTINEL_PATIENT_LIMIT = 8


def _find_project_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "run_full.py").is_file() and (candidate / "data" / "cards.json").is_file():
            return candidate
    raise FileNotFoundError("Project root must contain run_full.py and data/cards.json")


PROJECT_ROOT = _find_project_root()
CARDS_PATH = PROJECT_ROOT / "data" / "cards.json"
RUNNER_PATH = PROJECT_ROOT / "run_full.py"


def _validate_inputs(model_name: str, sample_count: int) -> tuple[str, int]:
    model_name = model_name.strip() if isinstance(model_name, str) else ""
    if not model_name:
        raise ValueError("MODEL_NAME 必须是 OpenAI 平台中的非空模型 ID")
    if isinstance(sample_count, bool) or not isinstance(sample_count, int):
        raise TypeError("SAMPLE_COUNT 必须是整数")

    total_patients = len(json.loads(CARDS_PATH.read_text(encoding="utf-8")))
    if not 1 <= sample_count <= total_patients:
        raise ValueError(f"SAMPLE_COUNT 必须在 1..{total_patients} 之间")
    return model_name, sample_count


def _patient_args(sample_count: int) -> list[str]:
    canonical = PROJECT_ROOT / "patients" / f"patients_n{sample_count}.txt"
    if sample_count in {100, 300} and canonical.is_file():
        return ["--rid-file", str(canonical)]
    return ["--n", str(sample_count)]


def _provider_document(model_name: str) -> tuple[str, dict]:
    arm_name = f"{model_name}@official"
    return arm_name, {
        "arms": {
            arm_name: {
                "name": arm_name,
                "provider": "openai_compat",
                "vendor": "openai",
                "endpoint": OPENAI_CHAT_COMPLETIONS_URL,
                "api_key_env": "OPENAI_API_KEY",
                "model_id": model_name,
                "send_temperature": False,
                "temperature": None,
                "send_max_tokens": False,
                "max_tokens": MAX_COMPLETION_TOKENS,
                "json_mode": True,
                "extra_body": {"max_completion_tokens": MAX_COMPLETION_TOKENS},
                "thinking": None,
                "price_in": 0.0,
                "price_out": 0.0,
                "default_workers": WORKERS,
            }
        }
    }


@contextmanager
def _temporary_provider_config(provider_document: dict):
    runtime_root = PROJECT_ROOT / "runs" / "_notebook_runtime"
    runtime_root.mkdir(parents=True, exist_ok=True)
    provider_path = runtime_root / f"openai_provider_{os.getpid()}_{uuid.uuid4().hex}.yaml"
    try:
        provider_path.write_text(
            yaml.safe_dump(provider_document, allow_unicode=True, sort_keys=False),
            encoding="utf-8",
        )
        yield provider_path
    finally:
        provider_path.unlink(missing_ok=True)
        try:
            runtime_root.rmdir()
        except OSError:
            pass


def _run_stage(command: list[str], env: dict[str, str], label: str) -> None:
    print(f"\n{'=' * 18} {label} {'=' * 18}", flush=True)
    subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)


def _latest_failure(model_name: str, *, smoke: bool) -> tuple[Path | None, list[str]]:
    root = PROJECT_ROOT / "runs"
    if smoke:
        root /= "_smoke"
    root /= "openai"
    candidates: list[tuple[str, Path, list[str]]] = []
    for meta_path in root.glob("*/run_meta.json"):
        try:
            payload = json.loads(meta_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        if payload.get("arm_config", {}).get("model_id") != model_name:
            continue
        for run in payload.get("runs", {}).values():
            if not isinstance(run, dict) or run.get("status") not in {"failed", "aborted"}:
                continue
            errors = [str(item) for item in run.get("errors", []) if item]
            timestamp = str(run.get("ended_at") or run.get("started_at") or "")
            candidates.append((timestamp, meta_path, errors))
    if not candidates:
        return None, []
    _, meta_path, errors = max(candidates, key=lambda item: item[0])
    return meta_path, errors


def _failure_hint(errors: list[str]) -> str:
    combined = "\n".join(errors).casefold()
    if "invalid_api_key" in combined or "incorrect api key" in combined or "authentication" in combined:
        return "提示：OpenAI key 无效或未获得目标 Project 的权限，请重新运行 key 单元。"
    if "model_not_found" in combined or "does not exist" in combined or "do not have access" in combined:
        return "提示：模型 ID 不存在，或当前 OpenAI Project 无权访问该模型。请使用平台显示的准确 API 模型 ID。"
    if "insufficient_quota" in combined or "billing" in combined:
        return "提示：OpenAI Project 没有可用额度或尚未启用 API 计费。"
    if "rate_limit" in combined or "rate limit" in combined or "http 429" in combined:
        return "提示：触发 OpenAI Project 的速率限制；已有成功结果会保留，稍后重新运行即可续跑。"
    if "unsupported_parameter" in combined or "response_format" in combined:
        return "提示：该模型不支持当前 Chat Completions/JSON 参数，请换用支持 Chat Completions 和 JSON mode 的 OpenAI 模型。"
    if "sslerror" in combined or "certificate" in combined:
        return "提示：这是本机 TLS/证书链问题，不是模型输出错误。"
    return "请查看上方服务端错误；检查模型 ID、Project 权限、API 余额和速率限制。"


def _raise_stage_error(label: str, model_name: str, exc: subprocess.CalledProcessError, *, smoke: bool) -> None:
    meta_path, errors = _latest_failure(model_name, smoke=smoke)
    details = "\n".join(f"- {item}" for item in errors[:3]) or "- 未在运行元数据中找到更详细的服务端错误，请查看上方子进程输出"
    location = str(meta_path) if meta_path else ("runs/_smoke/openai" if smoke else "runs/openai")
    suffix = "正式运行未启动。" if smoke else "已有成功记录已经保留；修复问题后重新运行会断点续跑。"
    raise RuntimeError(
        f"{label}失败（退出码 {exc.returncode}）。{suffix}\n"
        f"{details}\n{_failure_hint(errors)}\n完整记录位置：{location}"
    ) from None


def run_follow_up(model_name: str, sample_count: int) -> None:
    """使用 OpenAI 官方 API 运行固定 follow-up 实验。"""
    model_name, sample_count = _validate_inputs(model_name, sample_count)

    api_key = globals().get("OPENAI_API_KEY")
    if not isinstance(api_key, str) or not api_key.strip():
        raise RuntimeError("请先运行上方隐藏输入 OpenAI API key 的单元")

    env = os.environ.copy()
    env["OPENAI_API_KEY"] = api_key.strip()
    env["PYTHONIOENCODING"] = "utf-8"
    env["PYTHONUTF8"] = "1"

    arm_name, provider_document = _provider_document(model_name)
    with _temporary_provider_config(provider_document) as provider_path:

        command = [
            sys.executable,
            "-u",
            str(RUNNER_PATH),
            "--provider-config",
            str(provider_path),
            "--arm",
            arm_name,
            *_patient_args(sample_count),
            "--conds",
            CONDITIONS,
            "--repeats",
            str(REPEATS),
            "--month",
            str(MONTH),
            "--workers",
            str(WORKERS),
        ]

        main_calls = sample_count * MAIN_CONDITION_COUNT * REPEATS
        checkpoints = (main_calls + SENTINEL_EVERY - 1) // SENTINEL_EVERY
        sentinel_calls = checkpoints * min(sample_count, SENTINEL_PATIENT_LIMIT) * SENTINEL_CONDITION_COUNT
        print(f"模型 ID：{model_name}")
        print(f"患者数：{sample_count}")
        print(f"官方端点：{OPENAI_CHAT_COMPLETIONS_URL}")
        print(f"正式运行预计：{main_calls} 次主调用 + {sentinel_calls} 次 sentinel 调用")
        print("key 仅通过子进程环境变量传递，不会写入命令行、临时配置或结果元数据。")

        _run_stage([*command, "--plan"], env, "1/3 离线计划")
        try:
            _run_stage([*command, "--smoke", str(SMOKE_CALLS)], env, "2/3 冒烟测试")
        except subprocess.CalledProcessError as exc:
            _raise_stage_error("冒烟测试", model_name, exc, smoke=True)

        try:
            _run_stage(command, env, "3/3 正式运行 / 断点续跑")
        except subprocess.CalledProcessError as exc:
            _raise_stage_error("正式运行", model_name, exc, smoke=False)

    print(f"\n运行完成。结果已写入 runs/openai/{arm_name}/ 目录。")

In [ ]:
# 只修改下面两个值，然后运行本单元。
MODEL_NAME = "gpt-5.6-luna"
SAMPLE_COUNT = 300

run_follow_up(MODEL_NAME, SAMPLE_COUNT)